# Visualización de Datos de Automóviles con Matplotlib
## Aprendizaje Automático — Semana 2

Para esta actividad trabajé con el dataset **Automobile.csv**, que tiene información de 398 autos fabricados entre 1970 y 1982. Lo que me pareció interesante desde el principio es que el dataset mezcla autos de tres orígenes muy distintos (Estados Unidos, Europa y Japón) en una época donde la crisis del petróleo estaba cambiando todo el mercado automotriz, así que ya intuía que iba a encontrar diferencias importantes en el consumo.

La variable que quiero entender mejor es `mpg` (millas por galón), que básicamente mide qué tan eficiente es el auto. A continuación exploré las relaciones entre esa variable y las demás características del dataset usando distintos tipos de gráficos.

### Columnas del dataset:
| Columna | Descripción |
|---|---|
| `mpg` | Millas por galón (consumo de combustible) |
| `cylinders` | Cilindros en el motor |
| `displacement` | Desplazamiento del motor (pulgadas cúbicas) |
| `horsepower` | Potencia del motor (caballos de fuerza) |
| `weight` | Peso del automóvil |
| `acceleration` | Aceleración 0-60 mph (segundos) |
| `model_year` | Año del modelo |
| `origin` | Origen: usa / europe / japan |

## 1. Carga del conjunto de datos

Lo primero es cargar los datos y ver con qué estamos trabajando. Un detalle que noté enseguida: la columna `horsepower` tiene algunos registros vacíos, así que hay que convertirla a numérico con cuidado para no perder filas por error. También `origin` viene como texto (usa, europe, japan) en lugar de números, lo que en realidad hace el análisis más legible que tener un 1, 2 o 3 sin contexto.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# Cargar el dataset
df = pd.read_csv('archivos semana 2/Automobile.csv')

# Convertir horsepower a numérico (tiene valores faltantes representados como '')
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')

# Mapear origin a etiquetas legibles (ya viene como texto)
origin_map = {'usa': 1, 'europe': 2, 'japan': 3}
df['origin_num'] = df['origin'].map(origin_map)

print(f'Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
print(f'Valores nulos por columna:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.head()

In [ ]:
# Resumen estadístico
df.describe()

## 2. Gráfico de Dispersión: `mpg` vs `weight`

Lo primero que quise explorar fue la relación entre el peso del auto y su consumo. Mi hipótesis antes de graficar era bastante obvia: los autos más pesados deberían gastar más combustible. Pero lo interesante es ver qué tan fuerte es esa relación en los datos reales, y si los autos de distintos países se comportan diferente dentro de esa tendencia. Coloreé los puntos por origen para poder ver eso de una sola mirada.

In [ ]:
# Paleta de colores por origen
color_map = {'usa': '#e63946', 'europe': '#457b9d', 'japan': '#2a9d8f'}
colors = df['origin'].map(color_map)

fig, ax = plt.subplots(figsize=(9, 6))

for origen, grupo in df.groupby('origin'):
    ax.scatter(
        grupo['weight'], grupo['mpg'],
        c=color_map[origen], label=origen.capitalize(),
        alpha=0.7, edgecolors='white', linewidths=0.4, s=60
    )

# Línea de tendencia general
mask = df[['weight', 'mpg']].notna().all(axis=1)
z = np.polyfit(df.loc[mask, 'weight'], df.loc[mask, 'mpg'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['weight'].min(), df['weight'].max(), 200)
ax.plot(x_line, p(x_line), '--', color='gray', linewidth=1.5, label='Tendencia general')

ax.set_title('Relación entre Peso y Consumo de Combustible (MPG)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Peso del automóvil (lbs)', fontsize=12)
ax.set_ylabel('Consumo (MPG — Millas por Galón)', fontsize=12)
ax.legend(title='Origen', fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('scatter_mpg_weight.png', dpi=150, bbox_inches='tight')
plt.show()
print('Correlación (weight, mpg):', round(df['weight'].corr(df['mpg']), 4))

## 3. Histograma: Distribución del Consumo de Combustible (`mpg`)

Después de ver la relación con el peso, quería entender cómo se distribuye el consumo en general. ¿La mayoría de los autos del dataset son eficientes o ineficientes? ¿Hay muchos valores extremos en alguno de los dos extremos? Agregué líneas de media y mediana para ver si la distribución está equilibrada o si hay un sesgo claro.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

n, bins, patches = ax.hist(
    df['mpg'].dropna(), bins=20,
    color='#457b9d', edgecolor='white', linewidth=0.8, alpha=0.85
)

# Líneas de media y mediana
media  = df['mpg'].mean()
mediana = df['mpg'].median()
ax.axvline(media,   color='#e63946', linestyle='--', linewidth=1.8, label=f'Media: {media:.1f} mpg')
ax.axvline(mediana, color='#2a9d8f', linestyle='-',  linewidth=1.8, label=f'Mediana: {mediana:.1f} mpg')

ax.set_title('Distribución del Consumo de Combustible (MPG)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Consumo (MPG — Millas por Galón)', fontsize=12)
ax.set_ylabel('Frecuencia (Número de Automóviles)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('histograma_mpg.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Media: {media:.2f} | Mediana: {mediana:.2f} | Desv. estándar: {df["mpg"].std():.2f}')
print(f'Min: {df["mpg"].min()} | Max: {df["mpg"].max()}')

## 4. Diagrama de Caja (Boxplot): `cylinders` y `origin`

Para este gráfico quería comparar directamente cómo varía el consumo según el tipo de motor y según el país de origen. El boxplot es muy útil acá porque no solo muestra dónde está el centro de cada grupo, sino también qué tan dispersos están los datos y si hay autos que se salen mucho de lo esperado — los famosos **outliers**, que aparecen como puntos sueltos fuera de los bigotes. Los armé en paralelo para poder comparar las dos variables de un vistazo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# ── Boxplot por Cylinders ──
cilindros_unicos = sorted(df['cylinders'].dropna().unique())
datos_cil = [df.loc[df['cylinders'] == c, 'mpg'].dropna().values for c in cilindros_unicos]
colores_cil = ['#e63946', '#f4a261', '#e9c46a', '#2a9d8f', '#264653']

bp1 = axes[0].boxplot(
    datos_cil, patch_artist=True, notch=False,
    medianprops=dict(color='black', linewidth=2)
)
for patch, color in zip(bp1['boxes'], colores_cil):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

axes[0].set_xticklabels([f'{int(c)} cil.' for c in cilindros_unicos], fontsize=10)
axes[0].set_title('MPG por Número de Cilindros', fontsize=13, fontweight='bold', pad=10)
axes[0].set_xlabel('Cilindros', fontsize=11)
axes[0].set_ylabel('Consumo (MPG)', fontsize=11)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# ── Boxplot por Origin ──
origenes = ['usa', 'europe', 'japan']
etiquetas_origen = ['Estados Unidos', 'Europa', 'Japón']
datos_orig = [df.loc[df['origin'] == o, 'mpg'].dropna().values for o in origenes]
colores_orig = ['#e63946', '#457b9d', '#2a9d8f']

bp2 = axes[1].boxplot(
    datos_orig, patch_artist=True, notch=False,
    medianprops=dict(color='black', linewidth=2)
)
for patch, color in zip(bp2['boxes'], colores_orig):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

axes[1].set_xticklabels(etiquetas_origen, fontsize=10)
axes[1].set_title('MPG por Origen del Automóvil', fontsize=13, fontweight='bold', pad=10)
axes[1].set_xlabel('Origen', fontsize=11)
axes[1].set_ylabel('Consumo (MPG)', fontsize=11)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

fig.suptitle('Diagramas de Caja — Identificación de Valores Atípicos', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('boxplot_cylinders_origin.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Gráfico de Barras: Distribución por `model_year` y `origin`

Me surgió la pregunta: ¿la composición del mercado cambió con los años? Es decir, ¿siempre dominaron los autos americanos o hubo algún período donde Japón y Europa ganaron terreno? Este gráfico de barras agrupadas responde exactamente eso, mostrando cuántos autos de cada origen hay registrados por año de modelo.

In [ ]:
# Tabla pivote: filas=model_year, columnas=origin, valores=conteo
pivot = df.groupby(['model_year', 'origin']).size().unstack(fill_value=0)

# Asegurar orden de columnas
for col in ['usa', 'europe', 'japan']:
    if col not in pivot.columns:
        pivot[col] = 0
pivot = pivot[['usa', 'europe', 'japan']]

anios = pivot.index.astype(str)
x = np.arange(len(anios))
ancho = 0.25

fig, ax = plt.subplots(figsize=(13, 6))

barras_usa    = ax.bar(x - ancho, pivot['usa'],    ancho, label='Estados Unidos', color='#e63946', alpha=0.85)
barras_europe = ax.bar(x,         pivot['europe'], ancho, label='Europa',         color='#457b9d', alpha=0.85)
barras_japan  = ax.bar(x + ancho, pivot['japan'],  ancho, label='Japón',          color='#2a9d8f', alpha=0.85)

# Etiquetas de valor sobre cada barra
def etiquetar(barras):
    for barra in barras:
        h = barra.get_height()
        if h > 0:
            ax.text(
                barra.get_x() + barra.get_width() / 2, h + 0.15,
                str(int(h)), ha='center', va='bottom', fontsize=7.5
            )

etiquetar(barras_usa)
etiquetar(barras_europe)
etiquetar(barras_japan)

ax.set_title('Distribución de Automóviles por Año de Modelo y Origen', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Año del Modelo (19XX)', fontsize=12)
ax.set_ylabel('Cantidad de Automóviles', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(['19' + a for a in anios], rotation=45, ha='right', fontsize=10)
ax.legend(title='Origen', fontsize=10)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('barras_model_year_origin.png', dpi=150, bbox_inches='tight')
plt.show()

print('Total por origen:')
print(df['origin'].value_counts().rename({'usa': 'Estados Unidos', 'europe': 'Europa', 'japan': 'Japón'}))

## Reflexiones finales — Aporte al foro

Después de armar estos gráficos me quedo con varias cosas que me resultaron interesantes, y también con algunas reflexiones sobre por qué creo que este tipo de análisis es tan valioso antes de meterse de lleno al modelado.

**¿Por qué es importante dominar estas herramientas?**

Porque sin visualización estás prácticamente volando a ciegas. Los números en una tabla te dicen poco si no los podés interpretar rápido. En este ejercicio, la correlación entre `weight` y `mpg` es de -0.83 — un número que técnicamente es claro, pero que cuando lo ves graficado con los puntos separados por color según origen, de repente entendés no solo que hay correlación, sino *dónde* se agrupan los distintos fabricantes y *cuánto* se desvían de la tendencia general. Eso es algo que un número solo no te da.

**¿Cómo influyen en la comprensión, la comunicación y el éxito de un proyecto ML?**

En la comprensión, te ayudan a entender el problema antes de modelar. En la comunicación, son la diferencia entre explicarle algo a un colega o a un cliente que entiende vs. uno que no sabe nada de estadística — un gráfico bien hecho cruza esa barrera mucho mejor que una tabla de coeficientes. Y en el éxito del modelo, influyen directamente porque si no detectás outliers o distribuciones raras antes de entrenar, esos problemas van a aparecer después en el rendimiento del modelo, cuando ya es más difícil y costoso corregirlos.

**Anécdota: cómo un gráfico cambió mi enfoque en este mismo ejercicio**

Antes de graficar, yo asumiría que `acceleration` también iba a tener una relación fuerte con el consumo — tiene sentido intuitivo que un auto que acelera rápido gasta más. Pero cuando hice el scatter plot de `acceleration` vs `mpg` (lo probé antes de decidir qué mostrar acá), la nube de puntos no tenía ninguna forma clara. La correlación era casi nula. Eso me hizo replantear qué variables incluir en el análisis principal y me confirmó que sin graficar primero, hubiera incluido una variable poco informativa en un modelo futuro.

**¿Pueden las visualizaciones ser engañosas?**

Sí, y creo que es algo que hay que tener muy presente. Algunos casos típicos: si el eje Y no empieza en cero, una diferencia pequeña entre grupos puede parecer enorme visualmente. O si graficás solo la media sin mostrar la dispersión, podés concluir que dos grupos son iguales cuando en realidad uno tiene una variabilidad altísima. Otro riesgo clásico es confundir correlación con causalidad — el scatter de `weight` vs `mpg` muestra una relación muy clara, pero eso no significa que *alivianar* un auto garantice exactamente X unidades de mejora en el consumo; hay otras variables involucradas.

**¿Cómo garantizar que las visualizaciones sean precisas y efectivas?**

Algunas cosas que aprendí con la práctica: siempre etiquetar los ejes con unidades reales (no solo el nombre de la columna), incluir referencias estadísticas como la media o mediana cuando tiene sentido, mostrar la distribución completa y no solo resúmenes, y complementar siempre el gráfico con los números — como hace `df.describe()` o el print de la correlación. Un gráfico solo sin contexto numérico puede llevar a conclusiones erróneas, y un número solo sin gráfico puede ocultar patrones importantes. La combinación de ambos es lo que realmente da una imagen fiel de los datos.

Para cerrar: si tuviera que construir un modelo para predecir `mpg`, me apoyaría principalmente en `weight`, `cylinders` y `origin` como variables predictoras. Los gráficos lo dejan bastante claro, y eso es exactamente para lo que sirve el análisis exploratorio — darte una dirección antes de empezar a modelar.